# 08 · İstatistiksel Anlamlılık
**Tez:** Güneş Döngüleri 23–25 Boyunca Güneş Patlamaları, Güneş Lekeleri ve Jeomanyetik Etkiler
**Hazırlayan:** Tunahan İnci · **Defter sürümü:** 1.0 (10.11.2026)

Şimdiye kadarki bulguların rastlantıyla açıklanıp açıklanamayacağını test eder. Zaman serilerinde ardışık aylar birbirine bağlı olduğu için klasik testler (ör. Pearson korelasyonunun p-değeri) aşırı iyimser sonuç verir. Bu yüzden her testte **otokorelasyonu koruyan** yöntemler kullanılır.

| Bölüm | Soru | Yöntem |
|---|---|---|
| A | Döngü 25 gerçekten Döngü 24'ten daha verimli mi? | Döngü içinde 12 aylık blok bootstrap (5000 tekrar) |
| B | Leke sayısı ile patlama ve fırtına arasındaki ilişki ve gecikme anlamlı mı? | Faz rastgeleleştirilmiş yapay seriler (2000 adet), döngü bazında gecikme |
| C | Patlama sonrası fırtına olasılığındaki artış anlamlı mı? | Olay bootstrap'ı + 500 eşleştirilmiş rastgele küme; lojistik regresyon (doz–yanıt) |
| D | Konum farkı (merkez–doğu–batı) anlamlı mı? | Permütasyon testi |

Sonuçlar `08_anlamlilik.json` olarak kaydedilir; proje defterindeki "Sonuç adayları" bölümü bu dosyaya göre güncellenecek.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
KOK = Path('/content/drive/MyDrive/gunes_tezi')

In [ ]:
# Saf fonksiyonlar (test edilebilir)
import numpy as np, pandas as pd

def kumelenmeyi_ayikla(zaman, aki, pencere_saat=48):
    # Büyükten küçüğe giderek, kabul edilmiş bir olaya ±pencere içinde olanları atar.
    # Böylece aynı aktif dönemdeki ardışık patlamalar tek olay sayılır.
    sira = np.argsort(-np.asarray(aki))
    t = pd.to_datetime(pd.Series(zaman)).values.astype('datetime64[s]').astype(np.int64)
    kabul, kabul_t = [], []
    pen = pencere_saat * 3600
    for i in sira:
        if all(abs(t[i] - k) > pen for k in kabul_t):
            kabul.append(i); kabul_t.append(t[i])
    return np.sort(np.array(kabul, dtype=int))

def olay_matrisi(seri, zamanlar, once_saat=48, sonra_saat=120):
    # Saatlik seriden, her olay zamanının etrafındaki pencereyi satır olarak çıkarır.
    s = seri.sort_index()
    t0 = s.index[0]
    deger = s.values.astype(float)
    konum = ((pd.to_datetime(pd.Series(zamanlar)).dt.floor('h') - t0) / pd.Timedelta(hours=1)).values.astype(int)
    gecikme = np.arange(-once_saat, sonra_saat + 1)
    idx = konum[:, None] + gecikme[None, :]
    gecerli = (idx >= 0) & (idx < len(deger))
    M = np.full(idx.shape, np.nan)
    M[gecerli] = deger[idx[gecerli]]
    return gecikme, M

def rastgele_esle(zamanlar, alt, ust, gun=180, tekrar=1, tohum=0):
    # Her olay için ±gun içinde rastgele bir zaman seçer (güneş döngüsü evresini korur).
    rng = np.random.default_rng(tohum)
    z = pd.to_datetime(pd.Series(zamanlar)).values
    out = []
    for _ in range(tekrar):
        kayma = rng.uniform(-gun, gun, len(z)) * 86400
        r = z + (kayma * 1e9).astype('timedelta64[ns]')
        r = np.clip(r, np.datetime64(alt), np.datetime64(ust))
        out.append(pd.DatetimeIndex(r))
    return out

def firtina_var_mi(dst_saatlik, zamanlar, esik=-50, bas_saat=12, son_saat=96):
    gec, M = olay_matrisi(dst_saatlik, zamanlar, 0, son_saat)
    pencere = M[:, bas_saat:]
    return np.nanmin(pencere, axis=1) <= esik


# 08'e özgü saf fonksiyonlar (test edilebilir)
def blok_ornekle(n, blok, rng):
    # n uzunluğunda bir seri için dairesel hareketli blok bootstrap indeksleri
    k = int(np.ceil(n / blok))
    bas = rng.integers(0, n, k)
    return (bas[:, None] + np.arange(blok)[None, :]).ravel()[:n] % n

def dongu_bootstrap(ay, kolon, pencereler, tekrar=5000, blok=12, tohum=0):
    # Her döngü içinde 12 aylık bloklarla yeniden örnekleme; oran = Σkolon / ΣSN × 1000
    rng = np.random.default_rng(tohum)
    out = {}
    for d, (b, s) in pencereler.items():
        p = ay.loc[b:s]
        y, x = p[kolon].values.astype(float), p['SN'].values.astype(float)
        idx = np.array([blok_ornekle(len(p), blok, rng) for _ in range(tekrar)])
        out[d] = {'deger': y.sum() / x.sum() * 1000, 'boot': y[idx].sum(1) / x[idx].sum(1) * 1000}
    return out

def faz_rastgele(x, rng):
    # Güç spektrumunu (ve dolayısıyla otokorelasyonu) koruyan, fazları rastgele yapay seri
    x = np.asarray(x, float); m = x.mean()
    F = np.fft.rfft(x - m)
    faz = rng.uniform(0, 2 * np.pi, len(F)); faz[0] = 0
    if len(x) % 2 == 0: faz[-1] = 0
    return np.fft.irfft(np.abs(F) * np.exp(1j * faz), n=len(x)) + m

def gecikmeli_r(x, y, gecikmeler):
    # r(L) = korelasyon(x_t, y_{t+L}); pozitif L: y, x'in arkasından geliyor
    x = np.asarray(x, float); y = np.asarray(y, float); out = []
    for L in gecikmeler:
        a, b = (x[:len(x) - L], y[L:]) if L >= 0 else (x[-L:], y[:len(y) + L])
        out.append(np.corrcoef(a, b)[0, 1])
    return np.array(out)

def lojistik(x, y, iter_=50):
    # Tek değişkenli lojistik regresyon (Newton-Raphson). Dönüş: b0, b1, b1'in standart hatası
    X = np.column_stack([np.ones(len(x)), x]); b = np.zeros(2)
    for _ in range(iter_):
        p = 1 / (1 + np.exp(-X @ b)); W = p * (1 - p)
        H = X.T @ (X * W[:, None]); g = X.T @ (y - p)
        b = b + np.linalg.solve(H, g)
    p = 1 / (1 + np.exp(-X @ b)); H = X.T @ (X * (p * (1 - p))[:, None])
    return b[0], b[1], np.sqrt(np.linalg.inv(H)[1, 1])

In [ ]:
import json
import matplotlib.pyplot as plt
ISL, SEKIL = KOK/'data'/'processed', KOK/'figures'
fl = pd.read_parquet(ISL/'flare_final.parquet')
sn = pd.read_parquet(ISL/'sn_aylik_1749.parquet').set_index('tarih').sn
gunluk = pd.read_parquet(ISL/'omni2_gunluk.parquet').set_index('zaman')
omni = pd.read_parquet(ISL/'omni2_saatlik.parquet').set_index('zaman').sort_index()
omni = omni[~omni.index.duplicated()].asfreq('h')

idx = pd.date_range('2000-01-01', '2025-12-01', freq='MS')
ay = pd.DataFrame({
    'SN': sn.reindex(idx),
    'MX': fl[fl.aki >= 1e-5].set_index('pik').resample('MS').size().reindex(idx, fill_value=0),
    'X': fl[fl.aki >= 1e-4].set_index('pik').resample('MS').size().reindex(idx, fill_value=0),
    'F50': (gunluk.dst_min <= -50).resample('MS').sum().reindex(idx, fill_value=0),
    'F100': (gunluk.dst_min <= -100).resample('MS').sum().reindex(idx, fill_value=0),
})
DONGU = {'23': ('2000-01-01', '2008-11-01'), '24': ('2008-12-01', '2019-11-01'), '25': ('2019-12-01', '2025-12-01')}
SONUC = {}
print(ay.describe().round(1).to_string())

## A · Döngü verimliliği: güven aralıkları
Her döngüde 1000 leke başına olay sayısı ve %95 güven aralığı. Oranlar için tek yönlü p-değeri: bootstrap dağılımında oranın 1'in altına düşme sıklığı. Bloklar 12 ay, böylece ay içi ve mevsimsel bağımlılık korunur.

Not: Döngü 23 için S4 kalibrasyonu kullanılır. Bu testte kalibrasyon belirsizliği yoktur, yalnızca örnekleme belirsizliği vardır; köprü belirsizliği senaryolarla ayrıca raporlanmıştı.

In [ ]:
satir = []
SONUC['A'] = {}
for kolon, ad in [('MX', 'M+X'), ('X', 'X'), ('F100', 'Şiddetli fırtına günü'), ('F50', 'Orta+ fırtına günü')]:
    bs = dongu_bootstrap(ay, kolon, DONGU, 5000, 12, 1)
    for d, r in bs.items():
        lo, hi = np.percentile(r['boot'], [2.5, 97.5])
        satir.append([ad, f'D{d}', round(r['deger'], 1), round(lo, 1), round(hi, 1)])
    for a, b in [('25', '24'), ('23', '24'), ('25', '23')]:
        o = bs[a]['boot'] / bs[b]['boot']
        lo, hi = np.percentile(o, [2.5, 97.5])
        p = float((o <= 1).mean()) if bs[a]['deger'] >= bs[b]['deger'] else float((o >= 1).mean())
        satir.append([ad, f'D{a}/D{b}', round(bs[a]['deger'] / bs[b]['deger'], 3), round(lo, 3), round(hi, 3), round(p, 4)])
        SONUC['A'][f'{ad} D{a}/D{b}'] = {'oran': round(bs[a]['deger'] / bs[b]['deger'], 3), 'ga95': [round(lo, 3), round(hi, 3)], 'p_tek_yonlu': round(p, 4)}
tA = pd.DataFrame(satir, columns=['Ölçü', 'Döngü / oran', 'Değer', '%95 alt', '%95 üst', 'p (tek yönlü)'])
tA.to_csv(ISL/'08_A_dongu_guven.csv', index=False)
print(tA.to_string(index=False))
print('\nOkuma: Oran aralığı 1\'i içermiyorsa ve p < 0,05 ise fark örnekleme şansıyla açıklanamaz.')
print('Uyarı: Döngü 25 henüz bitmedi ve fırtınalar leke sayısını ~1 yıl geriden izliyor; D25 fırtına oranları eksik kalabilir.')

## B · İlişki ve gecikme
**B1.** Aylık (düzleştirilmemiş) serilerde gecikmeli korelasyon. Anlamlılık için leke serisinin 2000 faz rastgeleleştirilmiş kopyası üretilir: bunlar aynı otokorelasyona sahip ama gerçek ilişki taşımayan serilerdir. Gerçek korelasyon bu dağılımın %97,5'lik değerini aşarsa anlamlıdır. Tüm gecikmeleri birlikte test etmek için her yapay seride en yüksek korelasyon alınır (çoklu test düzeltmesi).

**B2.** Gecikmenin kararlılığı: 13 ay düzleştirilmiş serilerde en iyi gecikme her döngü için ayrı ayrı ve bir döngü dışarıda bırakılarak hesaplanır.

**B3.** Kısa vadeli bağ: döngü trendi (13 ay düzleştirme) çıkarıldıktan sonra kalan aylık dalgalanmalar arasında ilişki var mı?

In [ ]:
rng = np.random.default_rng(2)
L = np.arange(-36, 37)
x = ay.SN.values
SONUC['B'] = {}
fig, ax = plt.subplots(1, 3, figsize=(15, 4.2), sharey=True, constrained_layout=True)
for i, (kolon, ad, renk) in enumerate([('MX', 'M+X', '#3fa0c9'), ('F50', 'Orta+ fırtına günü', '#3451c4'), ('F100', 'Şiddetli fırtına günü', '#c4432c')]):
    y = ay[kolon].values
    r = gecikmeli_r(x, y, L)
    yapay = np.array([gecikmeli_r(faz_rastgele(x, rng), y, L) for _ in range(2000)])
    ust = np.percentile(yapay, 97.5, axis=0)
    maks = yapay.max(axis=1)
    en = int(np.argmax(r))
    p_tek = float((yapay[:, en] >= r[en]).mean())
    p_global = float((maks >= r[en]).mean())
    SONUC['B'][ad] = {'en_iyi_gecikme_ay': int(L[en]), 'r': round(float(r[en]), 3), 'r_gecikme0': round(float(r[36]), 3),
                      'p_tek_gecikme': round(p_tek, 4), 'p_global': round(p_global, 4)}
    ax[i].fill_between(L, np.percentile(yapay, 2.5, axis=0), ust, color='grey', alpha=.3, label='Yapay seriler %95')
    ax[i].plot(L, r, color=renk, lw=2, label='Gerçek'); ax[i].axvline(0, color='k', lw=.7)
    ax[i].scatter([L[en]], [r[en]], color=renk, zorder=3)
    ax[i].set_title(f'{ad}: en iyi gecikme {L[en]} ay, r = {r[en]:.2f}, p = {p_global:.3f}')
    ax[i].set_xlabel('Gecikme (ay)'); ax[i].grid(alpha=.25)
ax[0].set_ylabel('Korelasyon (aylık, düzleştirilmemiş)'); ax[0].legend(loc='lower left', fontsize=8)
fig.savefig(SEKIL/'08_B1_gecikme_anlamlilik.png', dpi=200); plt.show()
print(pd.DataFrame(SONUC['B']).T.to_string())

In [ ]:
duz = ay.rolling(13, center=True).mean()
Lg = np.arange(-30, 31)
def en_iyi(p, kolon):
    p = p.dropna(); r = gecikmeli_r(p.SN.values, p[kolon].values, Lg); i = int(np.nanargmax(r)); return int(Lg[i]), round(float(r[i]), 3)
satir = []
for kolon, ad in [('MX', 'M+X'), ('F50', 'Orta+ fırtına günü'), ('F100', 'Şiddetli fırtına günü')]:
    s = [ad, *en_iyi(duz, kolon)]
    for d, (b, e) in DONGU.items():
        s.append(en_iyi(duz.loc[pd.Timestamp(b) - pd.DateOffset(months=18): pd.Timestamp(e) + pd.DateOffset(months=18)], kolon)[0])
    for d, (b, e) in DONGU.items():
        disari = duz[(duz.index < b) | (duz.index > e)]
        s.append(en_iyi(disari, kolon)[0])
    satir.append(s)
tB2 = pd.DataFrame(satir, columns=['Seri', 'Tümü: gecikme', 'Tümü: r', 'D23', 'D24', 'D25', 'D23 hariç', 'D24 hariç', 'D25 hariç'])
tB2.to_csv(ISL/'08_B2_gecikme_kararlilik.csv', index=False)
print('B2 · En iyi gecikme (ay), döngü bazında ve bir döngü dışarıda:'); print(tB2.to_string(index=False))
SONUC['B2'] = tB2.to_dict(orient='records')

art = (ay - duz).dropna()
satir = []
for kolon, ad in [('MX', 'M+X'), ('F50', 'Orta+ fırtına günü'), ('F100', 'Şiddetli fırtına günü')]:
    for gec in [0, 1, 2]:
        r = gecikmeli_r(art.SN.values, art[kolon].values, [gec])[0]
        yap = np.array([gecikmeli_r(faz_rastgele(art.SN.values, rng), art[kolon].values, [gec])[0] for _ in range(1000)])
        satir.append([ad, gec, round(r, 3), round(float((np.abs(yap) >= abs(r)).mean()), 4)])
tB3 = pd.DataFrame(satir, columns=['Seri', 'Gecikme (ay)', 'r (trendden arındırılmış)', 'p (çift yönlü)'])
tB3.to_csv(ISL/'08_B3_kisa_vade.csv', index=False)
print('\nB3 · Trendden arındırılmış aylık dalgalanmalar arasındaki ilişki:'); print(tB3.to_string(index=False))
SONUC['B3'] = tB3.to_dict(orient='records')

## C · Patlama sonrası fırtına olasılığı
**C1.** 06 defterindeki gruplar aynen yeniden kurulur (48 saat kümelenme ayıklaması, 12–96 saat penceresi). Patlama sonrası olasılık için olay bootstrap'ı (5000), rastgele karşılaştırma için her olay ±180 gün içinden seçilen **500 bağımsız rastgele küme** kullanılır. p-değeri: 500 rastgele kümeden kaçının gerçek olasılığa ulaştığı.

**C2.** Doz–yanıt: M1 ve üstü tüm bağımsız olaylarda, fırtına olup olmamasının log10(pik akı) ile lojistik regresyonu. Pozitif ve anlamlı eğim: patlama büyüdükçe fırtına olasılığı artıyor.

In [ ]:
ALT, UST = '2000-01-03', '2025-12-25'
SINIF = {'M1–M5': (1e-5, 5e-5), 'M5–X1': (5e-5, 1e-4), '≥X1': (1e-4, 1)}
olay = {}
for ad, (a, b) in SINIF.items():
    s = fl[(fl.aki >= a) & (fl.aki < b) & (fl.pik >= ALT) & (fl.pik <= UST)]
    olay[ad] = s.iloc[kumelenmeyi_ayikla(s.pik, s.aki, 48)]
rng = np.random.default_rng(3)
satir = []; SONUC['C'] = {}
for ad, o in olay.items():
    for esik in (-50, -100):
        f = firtina_var_mi(omni.dst, o.pik, esik).astype(float)
        p = f.mean()
        boot = f[rng.integers(0, len(f), (5000, len(f)))].mean(1)
        rast = np.array([firtina_var_mi(omni.dst, r, esik).mean() for r in rastgele_esle(o.pik, ALT, UST, 180, 500, 11)])
        kat_boot = boot / rng.choice(rast, 5000)
        lo, hi = np.percentile(kat_boot, [2.5, 97.5])
        pdeg = float((rast >= p).mean())
        satir.append([ad, len(o), f'Dst ≤ {esik}', round(p * 100, 1), round(rast.mean() * 100, 1), round(p / rast.mean(), 2), round(lo, 2), round(hi, 2), pdeg])
        SONUC['C'][f'{ad} Dst≤{esik}'] = {'n': len(o), 'p_olay': round(p, 3), 'p_rastgele': round(float(rast.mean()), 3),
                                         'kat': round(p / rast.mean(), 2), 'kat_ga95': [round(lo, 2), round(hi, 2)], 'p': pdeg}
tC = pd.DataFrame(satir, columns=['Grup', 'n', 'Fırtına', 'Olay %', 'Rastgele %', 'Kat', 'Kat %95 alt', 'Kat %95 üst', 'p (500 rastgele küme)'])
tC.to_csv(ISL/'08_C1_firtina_anlamlilik.csv', index=False)
print(tC.to_string(index=False))
print('\np = 0 → 500 rastgele kümenin hiçbiri gerçek olasılığa ulaşamadı (p < 0,002).')

tum = pd.concat(olay.values())
tum = tum.iloc[kumelenmeyi_ayikla(tum.pik, tum.aki, 48)]
x = np.log10(tum.aki.values)
SONUC['C2'] = {}
for esik in (-50, -100):
    y = firtina_var_mi(omni.dst, tum.pik, esik).astype(float)
    b0, b1, se = lojistik(x, y)
    z = b1 / se
    from scipy.stats import norm
    pz = float(2 * norm.sf(abs(z)))
    SONUC['C2'][f'Dst≤{esik}'] = {'n': len(y), 'egim': round(b1, 3), 'se': round(se, 3), 'z': round(z, 2), 'p': pz,
                                  'odds_orani_10kat_aki': round(float(np.exp(b1)), 2)}
    print(f'C2 · Dst ≤ {esik}: n = {len(y)}, eğim = {b1:.3f} ± {se:.3f}, z = {z:.2f}, p = {pz:.2e}; '
          f'akı 10 kat artınca fırtına odds\'u {np.exp(b1):.2f} katına çıkıyor')

## D · Konum farkı
X sınıfı bağımsız olaylarda bölgeler arası şiddetli fırtına oranı farkı için permütasyon testi (10.000 karıştırma). Etiketler rastgele karıştırıldığında gözlenen fark kadar büyük bir farkın ne sıklıkla çıktığına bakılır.

In [ ]:
xo = olay['≥X1']
xo = xo[xo.bolge.isin(['doğu kenarı', 'merkez', 'batı kenarı'])].copy()
rng = np.random.default_rng(4)
satir = []; SONUC['D'] = {}
for esik in (-50, -100):
    xo['f'] = firtina_var_mi(omni.dst, xo.pik, esik).astype(float)
    for a, b in [('merkez', 'doğu kenarı'), ('merkez', 'batı kenarı'), ('batı kenarı', 'doğu kenarı')]:
        s = xo[xo.bolge.isin([a, b])]
        f, g = s.f.values, (s.bolge == a).values
        fark = f[g].mean() - f[~g].mean()
        perm = np.array([f[p].mean() - f[~p].mean() for p in (rng.permutation(g) for _ in range(10000))])
        pdeg = float((np.abs(perm) >= abs(fark)).mean())
        satir.append([f'Dst ≤ {esik}', f'{a} – {b}', int(g.sum()), int((~g).sum()), round(f[g].mean() * 100, 1), round(f[~g].mean() * 100, 1), round(fark * 100, 1), pdeg])
        SONUC['D'][f'Dst≤{esik} {a}–{b}'] = {'fark_yuzde_puan': round(fark * 100, 1), 'p': pdeg}
tD = pd.DataFrame(satir, columns=['Fırtına', 'Karşılaştırma', 'n1', 'n2', 'Oran1 %', 'Oran2 %', 'Fark (puan)', 'p (permütasyon)'])
tD.to_csv(ISL/'08_D_konum.csv', index=False)
print(tD.to_string(index=False))

def temizle(o):
    if isinstance(o, dict): return {str(k): temizle(v) for k, v in o.items()}
    if isinstance(o, list): return [temizle(v) for v in o]
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating,)): return float(o)
    return o
(ISL/'08_anlamlilik.json').write_text(json.dumps(temizle(SONUC), ensure_ascii=False, indent=1))
print('\nKaydedildi: 08_anlamlilik.json')